# 线性回归：机器学习的起点——拟合一条最佳直线
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：线性回归.py | 核心功能：最小二乘法拟合、评估指标详解、系数解读

## 概述

线性回归是机器学习的"Hello World"。假设目标变量 y 与特征 X 之间存在线性关系 y = Xw + b，通过最小化残差平方和（OLS，普通最小二乘法）求解权重 w 和截距 b。

别因为简单就小看它——线性回归在工业界仍然大量使用。房价预测、销量预估、广告转化率等场景，线性回归常常就是基线模型。它可解释性强（系数直接反映特征影响），训练速度极快，而且在很多实际问题上效果并不比复杂模型差多少。

脚本演示了线性回归的完整流程：数据生成、模型训练、预测评估、指标解读和系数分析。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 数学原理

### 1. 模型形式与矩阵表示

**代码对应**：`LinearRegression().fit(X_train, y_train)` 拟合线性模型。

多元线性回归模型为：

$$y_i = \mathbf{w}^T \mathbf{x}_i + b + \epsilon_i = \sum_{j=1}^{p} w_j x_{ij} + b + \epsilon_i$$

其中 $\mathbf{w} = (w_1, \ldots, w_p)^T$ 为系数向量，$b$ 为截距，$\epsilon_i \sim \mathcal{N}(0, \sigma^2)$ 为独立同分布的噪声。

矩阵形式（将截距合并）：$\mathbf{y} = \mathbf{X}\mathbf{w} + \boldsymbol{\epsilon}$，其中 $\mathbf{X} \in \mathbb{R}^{n \times (p+1)}$（第一列为全 1）。

### 2. 最小二乘法（OLS）推导

**代码对应**：sklearn 的 `LinearRegression` 内部使用 SVD 分解求解。

目标函数（残差平方和）：

$$J(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 = (\mathbf{y} - \mathbf{X}\mathbf{w})^T(\mathbf{y} - \mathbf{X}\mathbf{w})$$

展开：

$$J(\mathbf{w}) = \mathbf{y}^T\mathbf{y} - 2\mathbf{w}^T\mathbf{X}^T\mathbf{y} + \mathbf{w}^T\mathbf{X}^T\mathbf{X}\mathbf{w}$$

对 $\mathbf{w}$ 求梯度并令其为零：

$$\frac{\partial J}{\partial \mathbf{w}} = -2\mathbf{X}^T\mathbf{y} + 2\mathbf{X}^T\mathbf{X}\mathbf{w} = \mathbf{0}$$

得到**正规方程**（Normal Equation）：

$$\mathbf{X}^T\mathbf{X}\mathbf{w}^* = \mathbf{X}^T\mathbf{y}$$

当 $\mathbf{X}^T\mathbf{X}$ 可逆时，解为：

$$\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

**sklearn 的实现**：不直接求逆（数值不稳定），而是对 $\mathbf{X}$ 做 SVD 分解 $\mathbf{X} = \mathbf{U}\mathbf{\Sigma}\mathbf{V}^T$，则：

$$\mathbf{w}^* = \mathbf{V}\mathbf{\Sigma}^{-1}\mathbf{U}^T\mathbf{y}$$

### 3. 系数的统计性质

在 Gauss-Markov 假设下（线性、独立、同方差、无自相关），OLS 估计量具有以下性质：

**无偏性**：$\mathbb{E}[\hat{\mathbf{w}}] = \mathbf{w}$

**方差**：$\text{Var}(\hat{\mathbf{w}}) = \sigma^2 (\mathbf{X}^T\mathbf{X})^{-1}$

**BLUE（最佳线性无偏估计）**：在所有线性无偏估计量中，OLS 估计量的方差最小（Gauss-Markov 定理）。

**代码对应**：`lr.coef_` 返回 $\hat{\mathbf{w}}$，`lr.intercept_` 返回 $\hat{b}$。代码中的系数解读——"特征每增加 1 单位，$y$ 变化 $w_j$"——正是线性模型的直接含义。

### 4. 评估指标的数学定义

**代码对应**：代码中计算了 MSE、RMSE、MAE、R² 四种指标。

**均方误差（MSE）**：

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**均方根误差（RMSE）**：

$$\text{RMSE} = \sqrt{\text{MSE}}$$

RMSE 与 $y$ 同量纲，更直观。

**平均绝对误差（MAE）**：

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

MAE 对异常值更鲁棒（不像 MSE 那样放大残差）。

**决定系数 R²**：

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

**代码对应**：代码中手动计算 R² 的部分正是这个公式。$R^2 = 1$ 表示完美拟合，$R^2 = 0$ 表示模型等同于预测均值，$R^2 < 0$ 表示模型比均值还差。

### 5. 多重共线性与条件数

当特征之间高度相关时，$\mathbf{X}^T\mathbf{X}$ 接近奇异，条件数 $\kappa$ 很大：

$$\kappa(\mathbf{X}^T\mathbf{X}) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

条件数大意味着：系数估计的方差 $\text{Var}(\hat{w}_j) = \sigma^2 [(\mathbf{X}^T\mathbf{X})^{-1}]_{jj}$ 会很大，导致系数对数据微小变化极其敏感。这就是为什么多重共线性时需要正则化（Ridge、Lasso）。

### 6. 偏差-方差分解

线性回归的期望泛化误差可以分解为：

$$\mathbb{E}[(y - \hat{f}(\mathbf{x}))^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$$

其中：
- $\text{Bias}^2 = (\mathbb{E}[\hat{f}(\mathbf{x})] - f(\mathbf{x}))^2$：模型假设带来的系统误差
- $\text{Variance} = \mathbb{E}[(\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(\mathbf{x})])^2]$：对训练数据的敏感度
- $\sigma^2$：不可约噪声

线性回归的偏差取决于真实关系是否确实为线性。如果真实关系是非线性的，线性模型有不可消除的偏差。

## 代码结构

| 段落 | 内容 |
|------|------|
| 数据 | make_regression 生成 5 特征回归数据 |
| 模型训练 | LinearRegression 拟合 |
| 评估 | MSE、RMSE、MAE、R² 四种指标 |
| 系数解读 | 每个系数代表"特征每增加 1 单位，y 变化多少" |

### 1. 生成回归数据

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
X, y = make_regression(n_samples=200, n_features=5, n_informative=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 线性回归拟合

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)

print("=== 线性回归 ===")
print(f"系数: {lr.coef_.round(4)}")
print(f"截距: {lr.intercept_:.4f}")
print(f"\n训练集: MSE={mean_squared_error(y_train, y_pred_train):.4f}, "
      f"RMSE={np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}, "
# --- 继续 ---
      f"MAE={mean_absolute_error(y_train, y_pred_train):.4f}, "
      f"R²={r2_score(y_train, y_pred_train):.4f}")
print(f"测试集: MSE={mean_squared_error(y_test, y_pred_test):.4f}, "
      f"RMSE={np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}, "
      f"MAE={mean_absolute_error(y_test, y_pred_test):.4f}, "
# --- 继续 ---
      f"R²={r2_score(y_test, y_pred_test):.4f}")

### 3. 预测值 vs 真实值

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 前 10 个预测值 vs 真实值 ===")
for i in range(10):
    print(f"  真实={y_test[i]:>8.2f}, 预测={y_pred_test[i]:>8.2f}, 残差={y_test[i]-y_pred_test[i]:>8.2f}")

### 4. 评估指标详解

在测试集上评估模型性能，关注关键指标。

In [ ]:
print("\n=== 评估指标 ===")
print(f"MSE  (均方误差):     {mean_squared_error(y_test, y_pred_test):.4f}")
print(f"RMSE (均方根误差):   {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"MAE  (平均绝对误差): {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"R²   (决定系数):     {r2_score(y_test, y_pred_test):.4f}")

# 手动计算 R²
ss_res = np.sum((y_test - y_pred_test) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2_manual = 1 - ss_res / ss_tot
print(f"R² (手动计算): {r2_manual:.4f}")

### 5. 多元线性回归系数解读

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== 系数解读 ===")
print("每个系数表示：该特征每增加 1 个单位，目标变量平均变化多少")
for i, coef in enumerate(lr.coef_):
    print(f"  特征{i}: 系数={coef:.4f} (每增加 1 单位, y 变化 {coef:.4f})")

### 6. 注意事项

运行 6. 注意事项 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 线性回归要点 ===")
print("- 假设：线性关系、误差独立同分布、无多重共线性")
print("- 优点：简单可解释、训练速度快")
print("- 缺点：对异常值敏感、无法捕捉非线性关系")
print("- 如果特征间高度相关，考虑岭回归或 Lasso 回归")
# --- 输出结果 ---
print("- 如果存在非线性关系，考虑多项式回归或树模型")

## 关键代码解释

### R² 决定系数——最重要的回归指标

```python
ss_res = np.sum((y_test - y_pred_test) ** 2)  # 残差平方和
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)  # 总平方和
r2 = 1 - ss_res / ss_tot
```

R² 表示模型解释了多少数据的变异性。R² = 1 意味着完美拟合，R² = 0 意味着模型预测效果等于取均值。**负 R²** 意味着模型比"预测均值"还差——通常说明模型严重过拟合了训练集或数据分布不对。

### 系数解读

```python
for i, coef in enumerate(lr.coef_):
    print(f"特征{i}: 系数={coef:.4f}")
```

线性回归的系数直接可解释——如果特征 i 的系数是 3.5，意味着其他特征不变时，特征 i 每增加 1 个单位，y 平均增加 3.5。这种可解释性是很多复杂模型（如深度学习）不具备的。

## 使用示例

```python
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
print(f"R²: {lr.score(X_test, y_test):.4f}")
print(f"系数: {lr.coef_}")
```

## 注意事项

1. **线性假设**：如果 y 与 X 的关系是非线性的，线性回归会系统性地欠拟合
2. **多重共线性**：特征间高度相关时，系数估计不稳定（方差很大），考虑岭回归或 Lasso
3. **异常值敏感**：最小二乘法对异常值非常敏感（平方放大了异常值的影响）
4. **不需要特征缩放**：线性回归的 OLS 解不受特征尺度影响（但正则化版本需要）
5. **特征数 vs 样本数**：当特征数 > 样本数时，OLS 无唯一解，需要正则化

## 总结与延伸

以上代码展示了 **线性回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **梯度下降 vs 正规方程**：sklearn 的 LinearRegression 用 SVD 分解求解，比直接求逆矩阵更稳定
- **加权最小二乘（WLS）**：对异方差数据，可以给不同样本不同权重
- **广义线性模型（GLM）**：线性回归的推广——泊松回归（计数数据）、伽马回归（正偏态数据）
